#Initialisation

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType,DateType
from pyspark.sql.window import Window

#Read from Bronze

In [0]:

df=spark.table("workspace.bronze.crm_prd_info")

#Data transformation

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Product Key Parsing

In [0]:
df=df.withColumn("cat_dt",F.regexp_replace(F.substring(col("prd_key"),1,5),"-","_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

##Cost Cleanup

In [0]:
df=df.withColumn("prd_cost",F.coalesce(col("prd_cost"),F.lit(0)))

##Product Line Normalization

In [0]:
df = (
    df
    # Normalize product line
    .withColumn(
        "prd_line",
        F.when(F.upper(trim(col("prd_line"))) == "M", "Mountain")
         .when(F.upper(trim(col("prd_line"))) == "R", "Road")
         .when(F.upper(trim(col("prd_line"))) == "S", "Other Sales")
         .when(F.upper(trim(col("prd_line"))) == "T", "Touring")
         .otherwise("n/a")
    )
)
    

##Date casting

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))
df = df.withColumn("prd_end_dt", col("prd_end_dt").cast(DateType()))

#Renaming column

In [0]:
RENAME_MAP={
    "Prd_id":"Product_id",
    "Prd_key":"Product_key",
    "Prd_nm":"Product_name",
    "Prd_cost":"Product_cost",
    "Prd_line":"Product_line",
    "Prd_start_dt":"Product_start_date",
    "Prd_end_dt":"Product_end_date",
    "cat_dt":"Category_date"
}
for old_name,new_name in RENAME_MAP.items():
  df=df.withColumnRenamed(old_name,new_name)

In [0]:
df.limit(10).display()

#writing to silver table

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.crm_products")

In [0]:
%sql
select * from silver.crm_products